[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/altair-certified/notebooks/day-04-encodings-scales.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Encodings & Scales
**certified-journeys / altair-certified** · Day 4 · Encodings & Scales

> **Goal for today:** Master Altair's encoding type system (:Q/:O/:N/:T), apply custom axis formats and color schemes, and control scale domains and sort order.

In [ ]:
%pip install -q altair vega-datasets

## Step 1 · Encoding types: :Q vs :O vs :N vs :T

Altair uses a single-letter suffix to declare the *data type* of an encoding channel. The type changes **how the channel is scaled, what kind of axis appears, and how the legend looks**.

| Suffix | Name | Use when | Axis / legend |
|--------|------|----------|---------------|
| `:Q` | Quantitative | Continuous numbers (price, temperature) | Linear axis, continuous color ramp |
| `:O` | Ordinal | Discrete with natural order (ratings 1–5, months) | Categorical ticks, discrete legend |
| `:N` | Nominal | Unordered categories (country names, genres) | Categorical ticks, discrete legend, no sort |
| `:T` | Temporal | Dates and datetimes | Date-formatted axis |

The **same data field rendered with different types produces visually different charts** — choose the type that matches the semantics of your data, not just its storage format.

In [ ]:
import altair as alt
from vega_datasets import data
import pandas as pd

# Load a small dataset with numeric and temporal fields
cars = data.cars()

# Render the same 'Cylinders' field as :Q, :O, and :N side by side
base = alt.Chart(cars).mark_bar().encode(
    y=alt.Y('count()'),
)

chart_q = base.encode(x=alt.X('Cylinders:Q', title='Cylinders (Quantitative)')).properties(title=':Q — continuous')
chart_o = base.encode(x=alt.X('Cylinders:O', title='Cylinders (Ordinal)')).properties(title=':O — ordered discrete')
chart_n = base.encode(x=alt.X('Cylinders:N', title='Cylinders (Nominal)')).properties(title=':N — unordered discrete')

# Concatenate horizontally so we can compare axis tick behaviour
chart_q | chart_o | chart_n

### What just happened?

- **`:Q` axis** is a continuous number line — bars sit in bins, spacing reflects numeric distance.
- **`:O` axis** treats each value as a category but sorts numerically by default — equal-width bands.
- **`:N` axis** is identical in appearance to `:O` here, but with no implied order — useful when you don't want Altair to sort alphabetically or numerically.
- The key insight: **`:O` for small integer ranges** gives you discrete bands and a discrete legend, which is usually what you want for fields like star ratings or cylinder counts.

## Step 2 · Shorthand vs. longhand encoding

Altair supports two syntaxes for specifying an encoding channel:

```python
# Shorthand — concise, great for exploration
alt.X('Horsepower:Q')

# Longhand — verbose, required when adding titles/formats/scales
alt.X('Horsepower', type='quantitative', title='Engine Power (HP)')
```

You can also mix shorthand field+type with keyword arguments: `alt.X('Horsepower:Q', title='Engine Power (HP)')`. The longhand form is necessary when you need to pass an `alt.Axis()` or `alt.Scale()` object.

In [ ]:
# Compare shorthand vs longhand — both produce identical output
shorthand = alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N'
).properties(title='Shorthand', width=250)

longhand = alt.Chart(cars).mark_point().encode(
    x=alt.X('Horsepower', type='quantitative', title='Engine Power (HP)'),
    y=alt.Y('Miles_per_Gallon', type='quantitative', title='Fuel Economy (MPG)'),
    color=alt.Color('Origin', type='nominal', title='Country')
).properties(title='Longhand with custom titles', width=250)

shorthand | longhand

### What just happened?

- **Shorthand** (`'Horsepower:Q'`) uses the field name as the axis title automatically.
- **Longhand** lets you supply a human-readable `title=` on the encoding itself — this is the recommended approach for production charts.
- Both charts are structurally identical in the Vega-Lite JSON; shorthand is just syntactic sugar that Altair expands before serialising.

## Step 3 · Axis titles, number formats, and date formats

Use `alt.Axis()` to customise tick labels. The `format` parameter accepts **d3-format strings** for numbers and **d3-time-format strings** for dates.

Common format strings:

| Format | Produces | Use for |
|--------|----------|---------|
| `".2f"` | `3.14` | Fixed 2 decimal places |
| `".0%"` | `75%` | Percentage with no decimals |
| `",.0f"` | `1,234` | Thousands separator |
| `"%b %Y"` | `Jun 2024` | Month + year for temporal axes |

In [ ]:
# Axis formatting: number format on Y, date format on X
seattle = data.seattle_weather()

alt.Chart(seattle).mark_line().encode(
    x=alt.X('date:T',
            title='Month',
            axis=alt.Axis(format='%b %Y', labelAngle=-30)),  # temporal format
    y=alt.Y('mean(temp_max):Q',
            title='Avg Max Temp (°C)',
            axis=alt.Axis(format='.1f'))                     # numeric format: 1 decimal
).properties(title='Seattle Average Max Temperature', width=500)

### What just happened?

- `axis=alt.Axis(format=...)` controls the **tick label format** — not the data itself.
- `format='%b %Y'` formats dates as abbreviated month + year (`Jun 2024`).
- `format='.1f'` shows exactly one decimal on numeric ticks.
- **`labelAngle=-30`** rotates date labels to prevent overlap — a common necessity for time-series.

## Step 4 · Named color schemes with alt.Scale

Altair delegates color mapping to Vega/Vega-Lite, which supports all [Vega color schemes](https://vega.github.io/vega/docs/schemes/). Apply a named scheme with `alt.Scale(scheme='...')`.

Popular schemes:

| Scheme | Use for |
|--------|---------|
| `'viridis'` | Sequential quantitative, perceptually uniform |
| `'plasma'` | Sequential quantitative, high contrast |
| `'tableau10'` | Categorical (up to 10 classes) |
| `'redyellowblue'` | Diverging (positive/negative values) |

In [ ]:
# Apply the 'viridis' sequential color scheme to a heatmap
seattle_monthly = seattle.copy()
seattle_monthly['month'] = pd.to_datetime(seattle_monthly['date']).dt.month
seattle_monthly['year']  = pd.to_datetime(seattle_monthly['date']).dt.year

alt.Chart(seattle_monthly).mark_rect().encode(
    x=alt.X('month:O', title='Month', axis=alt.Axis(labelExpr="datum.value + ''")),
    y=alt.Y('year:O', title='Year'),
    color=alt.Color('mean(temp_max):Q',
                    title='Avg Max Temp',
                    scale=alt.Scale(scheme='viridis'))  # named color scheme
).properties(title='Seattle Temps — Viridis Scheme', width=380, height=220)

### What just happened?

- `alt.Scale(scheme='viridis')` maps the quantitative temperature range to the viridis color ramp.
- **Viridis is perceptually uniform** — equal value steps produce equal perceived color differences, making it safe for colorblind viewers.
- You can override just the scheme without touching domain or range — Altair infers min/max from the data automatically.

## Step 5 · Stack encoding for 100% stacked bars

The `stack` encoding property controls how bar heights accumulate:

| `stack=` value | Effect |
|----------------|--------|
| `True` (default) | Absolute stacked heights |
| `'normalize'` | 100% stacked (proportional) |
| `False` / `None` | Overlapping / layered bars |

In [ ]:
# 100% stacked bar: proportion of car origins per cylinder count
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Cylinders:O', title='Cylinders'),
    y=alt.Y('count():Q',
            stack='normalize',              # key: normalise to 100%
            axis=alt.Axis(format='.0%'),    # show as percentage on Y axis
            title='Share of Cars'),
    color=alt.Color('Origin:N', title='Country of Origin')
).properties(title='Car Origins by Cylinder Count (100% stacked)', width=350)

### What just happened?

- `stack='normalize'` rescales each bar to sum to 1 (100%), showing composition rather than total count.
- Combined with `format='.0%'` on the Y axis, ticks display as `0%`, `25%`, `50%` etc.
- **Use 100% stacked when you care about proportion**, not volume — it loses absolute count information.

## Step 6 · Sort encoding — ordering bars by value

The `sort` parameter on an encoding channel controls bar / point order. Accepted values:

| `sort=` | Behaviour |
|---------|----------|
| `'-y'` | Descending by the Y channel's aggregated value |
| `'y'` | Ascending by Y channel value |
| `'x'` | Ascending by X field value (alphabetical / numeric) |
| `alt.EncodingSortField(...)` | Full control: field, op, order |

In [ ]:
# Sort bars by descending mean horsepower — no pre-sorting in pandas needed
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Origin:N',
            sort='-y',          # sort X categories by descending Y aggregate
            title='Country of Origin'),
    y=alt.Y('mean(Horsepower):Q',
            title='Mean Horsepower',
            axis=alt.Axis(format='.0f')),
    color=alt.Color('Origin:N', legend=None)
).properties(title='Average Horsepower by Origin (sorted descending)', width=300)

### What just happened?

- `sort='-y'` tells Altair to **sort the X nominal axis by descending Y value** — the aggregation happens inside Vega-Lite, not in Python.
- The minus prefix means descending; `sort='y'` would be ascending.
- **No pre-sorting in pandas is needed** — this keeps your source data unchanged and makes the chart declarative.

## Step 7 · Clamping a scale with alt.Scale(domain=...)

By default, Altair sets scale domain from the data min/max. You can override with `alt.Scale(domain=[min, max])` to:

- **Clamp** to a meaningful range (e.g. 0–100 for scores)
- **Align** multiple charts to the same axis range
- **Zoom in** on a region of interest without filtering the underlying data

In [ ]:
# Clamp the Y axis to [0, 100] regardless of data range
auto_domain = alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y=alt.Y('Acceleration:Q', title='Acceleration')
).properties(title='Auto domain', width=220)

clamped_domain = alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y=alt.Y('Acceleration:Q',
            title='Acceleration',
            scale=alt.Scale(domain=[0, 100]))  # force axis to 0–100
).properties(title='Clamped to [0, 100]', width=220)

auto_domain | clamped_domain

### What just happened?

- `alt.Scale(domain=[0, 100])` pins the Y axis between 0 and 100 regardless of data values.
- Points outside the domain are **clipped** at the boundary — they do not vanish from the data, just from the visible area.
- This is especially useful when comparing two charts: **set the same domain on both** to make visual comparisons fair.

In [ ]:
# Challenge: Putting it all together
# Using the 'cars' dataset:
#   1. Create a bar chart of mean Miles_per_Gallon by Origin
#   2. Sort bars by descending Y value
#   3. Apply longhand encoding with a human-readable Y-axis title
#   4. Format Y-axis ticks to 1 decimal place
#   5. Apply the 'tableau10' color scheme to the color channel
#   6. Clamp the Y scale domain to [0, 40]

# Your solution here:
# challenge = alt.Chart(cars).mark_bar().encode(
#     x=alt.X('Origin:N', sort=___),
#     y=alt.Y(___, scale=alt.Scale(___), axis=alt.Axis(___)),
#     color=alt.Color('Origin:N', scale=alt.Scale(scheme=___), legend=None)
# ).properties(title='MPG by Origin', width=300)
# challenge

---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `:Q` vs `:O` | Q is continuous (number line); O is discrete ordered (bands). Same data, different visual. |
| `:N` vs `:O` | Both categorical, but `:O` implies a sort order; `:N` does not. |
| Shorthand vs longhand | Shorthand for exploration; longhand when you need `alt.Axis()` or `alt.Scale()`. |
| `axis=alt.Axis(format=...)` | Controls tick label format — d3-format for numbers, d3-time-format for dates. |
| `scale=alt.Scale(scheme=...)` | Named Vega color schemes; viridis for sequential, tableau10 for categorical. |
| `stack='normalize'` | 100% stacked bars — shows proportion, not volume. |
| `sort='-y'` | Sort X nominal axis by descending Y aggregate — no pandas pre-sort needed. |
| `scale=alt.Scale(domain=[a,b])` | Pin the axis range; clips out-of-range points visually. |

> **Tip:** Use `:O` for small integer ranges you want treated as discrete — it changes axis ticks, legend type, and default sort behavior compared to `:Q`.

---
## What's next
**Day 5** → Transforms — filter, calculate, aggregate, bin, window, and fold — all inside Vega-Lite without touching pandas.

Mark Day 4 complete in your [tracker](../index.html).